<a href="https://colab.research.google.com/github/abilashkannanv/AIML/blob/main/TrustCart_Phase2_Fake_Review_Detection1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/sample_data/reviews.csv')

# Display the first 5 rows to understand the data structure
display(df.head())

,review_text,seller_id,review_type
0,Once enjoying my crisp cool water. Our orders ...,4,genuine
1,This should say it all: we found a dress we lo...,1,genuine
2,Slow service\nBelow average food \nIll pass,2,genuine
3,I hit the Primanti Brothers Market Square loca...,3,genuine
4,An impressive and thoughtfully designed produc...,5,fake_generated


### Inspecting Data Structure and Distribution

In [37]:
# Get a concise summary of the DataFrame, including data types and non-null values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_text  6000 non-null   object
 1   seller_id    6000 non-null   int64 
 2   review_type  6000 non-null   object
dtypes: int64(1), object(2)
memory usage: 140.8+ KB


In [38]:
# Get descriptive statistics for numerical columns
display(df.describe())

,seller_id
count,6000.000000
mean,3.486500
std,1.362153
min,1.000000
25%,2.000000
50%,4.000000
75%,5.000000
max,5.000000


In [39]:
# Check for missing values
display(df.isnull().sum())

,0
review_text,0
seller_id,0
review_type,0


### Convert `review_type` into a binary target variable

In [40]:
# Display the unique values and their counts for 'review_type' before conversion
print("Original 'review_type' distribution:")
print(df['review_type'].value_counts())

# Convert 'genuine' to 1 (positive) and 'fake_generated'/'fake_templated' to 0 (negative)
df['review_type_binary'] = df['review_type'].map({'genuine': 1, 'fake_generated': 0, 'fake_templated': 0})

# Display the unique values and their counts for the new binary column
print("\nBinary 'review_type_binary' distribution:")
print(df['review_type_binary'].value_counts())

# Display the head with the new column
display(df.head())

Original 'review_type' distribution:
review_type
genuine           4000
fake_generated    1000
fake_templated    1000
Name: count, dtype: int64

Binary 'review_type_binary' distribution:
review_type_binary
1    4000
0    2000
Name: count, dtype: int64


,review_text,seller_id,review_type,review_type_binary
0,Once enjoying my crisp cool water. Our orders ...,4,genuine,1
1,This should say it all: we found a dress we lo...,1,genuine,1
2,Slow service\nBelow average food \nIll pass,2,genuine,1
3,I hit the Primanti Brothers Market Square loca...,3,genuine,1
4,An impressive and thoughtfully designed produc...,5,fake_generated,0


### Identify input features and target labels

In [41]:
# The target variable is 'review_type_binary'
y = df['review_type_binary']

# Input features will be 'review_text'. We will process this text later.
X = df['review_text']

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print("\nFirst 5 target labels:")
print(y.head())

Features (X) shape: (6000,)
Target (y) shape: (6000,)

First 5 target labels:
0    1
1    1
2    1
3    1
4    0
Name: review_type_binary, dtype: int64


## Task 2: Text Preprocessing

We will preprocess the `review_text` by converting it to lowercase, removing punctuation, numbers, and special characters, and normalizing whitespace. This helps in standardizing the text data for analysis.

In [42]:
import re

# Convert text to lowercase
X_cleaned = X.str.lower()

print("Original X (first 5):")
print(X.head())
print("\nLowercase X_cleaned (first 5):")
print(X_cleaned.head())

Original X (first 5):
0    Once enjoying my crisp cool water. Our orders ...
1    This should say it all: we found a dress we lo...
2          Slow service\nBelow average food \nIll pass
3    I hit the Primanti Brothers Market Square loca...
4    An impressive and thoughtfully designed produc...
Name: review_text, dtype: object

Lowercase X_cleaned (first 5):
0    once enjoying my crisp cool water. our orders ...
1    this should say it all: we found a dress we lo...
2          slow service\nbelow average food \nill pass
3    i hit the primanti brothers market square loca...
4    an impressive and thoughtfully designed produc...
Name: review_text, dtype: object


In [43]:
# Remove punctuation, numbers, and special characters
# Keep only alphabetic characters and spaces
X_cleaned = X_cleaned.apply(lambda text: re.sub(r'[^a-z\s]', '', text))

print("X after removing punctuation, numbers, and special characters (first 5):")
print(X_cleaned.head())

X after removing punctuation, numbers, and special characters (first 5):
0    once enjoying my crisp cool water our orders w...
1    this should say it all we found a dress we lov...
2            slow servicenbelow average food nill pass
3    i hit the primanti brothers market square loca...
4    an impressive and thoughtfully designed produc...
Name: review_text, dtype: object


In [44]:
# Normalize whitespace (remove extra spaces, tabs, newlines)
X_cleaned = X_cleaned.apply(lambda text: re.sub(r'\s+', ' ', text).strip())

print("X after normalizing whitespace (first 5):")
print(X_cleaned.head())

X after normalizing whitespace (first 5):
0    once enjoying my crisp cool water our orders w...
1    this should say it all we found a dress we lov...
2            slow servicenbelow average food nill pass
3    i hit the primanti brothers market square loca...
4    an impressive and thoughtfully designed produc...
Name: review_text, dtype: object


The `X_cleaned` Series now contains the preprocessed text, ready for tokenization. This will serve as our cleaned input features for further steps like vectorization.

## Task 3: Tokenization & Padding

We will now prepare the cleaned text for model input by converting it into numerical sequences and ensuring uniform length through padding.

In [45]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Initialize the tokenizer
# num_words: the maximum number of words to keep, based on word frequency.
# oov_token: a token to represent out-of-vocabulary words.
tokenizer = Tokenizer(num_words=10000, oov_token="<unk>")

# Fit the tokenizer on the cleaned text data
tokenizer.fit_on_texts(X_cleaned)

# Convert text to sequences of integers
X_sequences = tokenizer.texts_to_sequences(X_cleaned)

print(f"Original text (first entry): {X_cleaned.iloc[0]}")
print(f"Tokenized sequence (first entry): {X_sequences[0][:10]}...") # show first 10 tokens
print(f"Vocabulary size: {len(tokenizer.word_index)}")

Original text (first entry): once enjoying my crisp cool water our orders were taken promptly i ordered the chicken club and customized it with a side of coleslaw my better half had the fish sandwich with a coleslaw as well the food came like the wind the coleslaw was damn good the best our sandwiches couldnt have been better if jesus himself made them all around it was the bees knees
Tokenized sequence (first entry): [320, 1906, 16, 1344, 551, 441, 50, 645, 26, 722]...
Vocabulary size: 23944


In [46]:
# Determine the maximum sequence length
max_sequence_length = max([len(x) for x in X_sequences])
print(f"Maximum sequence length: {max_sequence_length}")

# Apply padding to ensure uniform input length
# padding='post' means padding is added at the end of sequences
# truncating='post' means truncation happens at the end of sequences
X_padded = pad_sequences(X_sequences, maxlen=max_sequence_length, padding='post', truncating='post')

print(f"\nShape of X_padded: {X_padded.shape}")
print(f"First padded sequence (first 10 elements): {X_padded[0][:10]}...")

Maximum sequence length: 932

Shape of X_padded: (6000, 932)
First padded sequence (first 10 elements): [ 320 1906   16 1344  551  441   50  645   26  722]...


The `X_padded` array now contains the numerical, padded sequences of your review text, ready to be used as input for a neural network.

## Task 4: Train Test Split with Leakage Prevention

In [47]:
from sklearn.model_selection import GroupShuffleSplit

# Assign X_padded to X for input features
X = X_padded

# Assign the binary target variable to y
y = df["review_type_binary"].values

# Assign 'seller_id' as the grouping variable to prevent leakage
groups = df["seller_id"]

# Initialize GroupShuffleSplit
# test_size=0.2 means 20% of the data will be used for testing
# random_state ensures reproducibility
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# Perform the group-based split
train_idx, test_idx = next(gss.split(X, y, groups))

# Create the training and testing sets
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (5100, 932)
X_test shape: (900, 932)
y_train shape: (5100,)
y_test shape: (900,)


### Explanation: Why Leakage Prevention is Critical in Real-World ML Systems

Data leakage occurs when information from the test set (or other unseen data) is inadvertently used during the training of a machine learning model. This leads to an over-optimistic evaluation of the model's performance on unseen data, as the model has essentially 'cheated' by seeing some aspect of the answers during training.

In this specific scenario, using `seller_id` for group-based splitting is crucial because:

1.  **Maintaining Realism**: In a real-world application, if we're trying to detect fake reviews, a model should be able to generalize to reviews from *new* sellers it hasn't seen before. If reviews from the same `seller_id` appear in both the training and test sets, the model might learn seller-specific patterns that are not indicative of its ability to detect fake reviews universally.
2.  **Avoiding Overfitting**: Without group-based splitting, the model might overfit to the characteristics of specific sellers. For example, if a particular seller always generates fake reviews in a very distinct style, and that seller's reviews are present in both train and test sets, the model might learn to associate that style directly with 'fake' rather than learning more generalized features of fake reviews.
3.  **Reliable Evaluation**: The goal of a test set is to simulate how well the model will perform on truly unseen data. If there's leakage, the test set doesn't accurately represent unseen data, and the reported performance metrics (e.g., accuracy, precision, recall) will be misleadingly high. This can lead to deploying a model that performs poorly in production.

By ensuring that all reviews from a given `seller_id` are either entirely in the training set or entirely in the test set, we prevent the model from learning seller-specific biases and force it to learn more generalizable patterns for identifying genuine versus fake reviews. This provides a more robust and realistic assessment of the model's true performance.

## Task 5: Model Architecture Design

### Loading GloVe Embeddings

To use GloVe embeddings, you'll first need to download them. A common source is the Stanford NLP website (e.g., `glove.6B.100d.txt`).

**Instructions to download GloVe:**

1.  **Download**: Visit [https://nlp.stanford.edu/projects/glove/](https://nlp.stanford.edu/projects/glove/) and download the desired pre-trained vectors (e.g., `glove.6B.zip`).
2.  **Upload to Colab**: Unzip the file and upload `glove.6B.100d.txt` to your Colab environment or Google Drive and mount it. For this example, we'll assume it's directly accessible in the Colab environment (e.g., in `/content/glove.6B.100d.txt`).

The following code will load the GloVe embeddings into a dictionary.

In [48]:
import numpy as np

# Define the path to your GloVe file
# Make sure to upload 'glove.6B.100d.txt' to your Colab environment
glove_path = '/content/glove.6B.100d.txt' # Adjust this path if necessary

# Determine the embedding dimension from the filename (e.g., 100 for 100d)
EMBEDDING_DIM = 100 # This should match the GloVe file you downloaded (e.g., 100 for glove.6B.100d.txt)

embeddings_index = {}
try:
    with open(glove_path, encoding='utf8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
    print(f'Found {len(embeddings_index)} word vectors in GloVe.')
except FileNotFoundError:
    print(f"Error: GloVe file not found at {glove_path}. Please download and upload the file as instructed above.")
    print("Proceeding with a placeholder embedding index for demonstration.")
    # Placeholder if GloVe file is not found - for demonstration purposes
    embeddings_index['test'] = np.random.rand(EMBEDDING_DIM)
    embeddings_index['word'] = np.random.rand(EMBEDDING_DIM)

Error: GloVe file not found at /content/glove.6B.100d.txt. Please download and upload the file as instructed above.
Proceeding with a placeholder embedding index for demonstration.


### Creating the Embedding Matrix

Now we will create an embedding matrix that will be used in our Keras Embedding layer. This matrix will contain the GloVe vector for each word in our tokenizer's vocabulary. Words not found in GloVe will be initialized with random vectors or zeros.

In [49]:
# Get the word index from the tokenizer
word_index = tokenizer.word_index
VOCAB_SIZE = len(word_index) + 1 # +1 for padding/unknown words

# Create a zero matrix for the embeddings
embedding_matrix = np.zeros((VOCAB_SIZE, EMBEDDING_DIM))

# Fill the embedding matrix with GloVe vectors
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        # Words not found in embedding index will be all-zeros.
        embedding_matrix[i] = embedding_vector
    else:
        # Optionally, initialize with random values for OOV words
        embedding_matrix[i] = np.random.rand(EMBEDDING_DIM)

print(f"Embedding matrix shape: {embedding_matrix.shape}")

Embedding matrix shape: (23945, 100)


### Model Architecture and Compilation

We will now define the model with an embedding layer, a bidirectional LSTM, and a dense output layer. The model will then be compiled with binary cross-entropy loss and the Adam optimizer.

In [50]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Define the model
model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,          # Size of the vocabulary
        output_dim=EMBEDDING_DIM,      # Dimension of the word embeddings
        weights=[embedding_matrix],    # Pre-trained GloVe embeddings
        input_length=max_sequence_length, # Length of input sequences (932 from X_padded)
        trainable=False                # Set to False to keep embeddings fixed
    ),
    Bidirectional(LSTM(units=128, return_sequences=False)), # Bidirectional LSTM layer
    Dropout(0.5), # Add dropout for regularization
    Dense(1, activation='sigmoid')     # Output layer for binary classification
])

# Compile the model
model.compile(
    loss='binary_crossentropy',        # Appropriate loss for binary classification
    optimizer='adam',                  # Adam optimizer
    metrics=['accuracy']               # Monitor accuracy
)

# Display the model summary
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     2,394,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,394,500 (9.13 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,394,500 (9.13 MB)

### Model Training with Early Stopping

Now we will train the model using `X_train`, `y_train` and validate it using `X_test`, `y_test`. We'll incorporate Early Stopping to prevent overfitting, monitoring the validation loss.

In [ ]:
# Define Early Stopping callback
# Monitor 'val_loss' and stop training if it doesn't improve for 3 epochs (patience=3)
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model
history = model.fit(
    X_train,
    y_train,
    epochs=5,                   # Run for a maximum of 5 epochs as specified
    batch_size=16,              # Sequence size of 16 as specified (interpreted as batch_size)
    validation_data=(X_test, y_test),
    callbacks=[early_stopping]  # Apply Early Stopping
)

print("\nModel training complete.")

Epoch 1/5
319/319 ━━━━━━━━━━━━━━━━━━━━ 388s 1s/step - accuracy: 0.9431 - loss: 0.1545 - val_accuracy: 0.9756 - val_loss: 0.0549
Epoch 2/5
216/319 ━━━━━━━━━━━━━━━━━━━━ 1:59 1s/step - accuracy: 0.9791 - loss: 0.0645